# Batch xenofobia — Parte 1: Subir

Este notebook sube los batches que faltan o fallaron.  
**Al final imprime los IDs nuevos** — copialos al notebook de descarga.

No hace polling. Solo sube y termina.

In [1]:
import os, json
from pathlib import Path
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
print('Cliente OK')

Cliente OK


In [2]:
INPUT_CSV         = 'DNU_data/tweets_clasificados.csv'
TEXT_COLUMN       = 'text'
BATCH_SIZE        = 50_000
MODEL_NAME        = 'gpt-4.1-mini'
TEMPERATURE       = 0
MAX_TOKENS        = 20
COMPLETION_WINDOW = '24h'
JSONL_DIR         = Path('outs_temp')

In [3]:
# IDs conocidos de la corrida anterior.
# None = nunca enviado / necesita reenvío.
BATCH_REGISTRY = {
    1: 'batch_69d055d51dac8190b74033f1192fe9b9',  # completed 50k/50k
    2: 'batch_69d055ff99ac819092b4cdbc0316d286',  # completed 50k/50k
    3: 'batch_69d0562a9bd081908c953e14b591dcf5',  # completed 50k/50k
    4: 'batch_69d05655b2b0819089abbed186402091',  # completed 50k/50k
    5: 'batch_69d0567f64908190842c44876532c5bb',  # failed → se reenvía
    6: 'batch_69d05688d95481909361e6dfc7d7e688',  # failed → se reenvía
}

In [4]:
df = pd.read_csv(INPUT_CSV)
df.index.name = 'row_id'
df = df.reset_index()
print(f'Filas cargadas: {len(df):,}')

Filas cargadas: 259,752


In [5]:
SYSTEM_MESSAGE = (
    'You are a classifier of xenophobic speech.\n\n'
    'Your task is to determine whether a text contains xenophobic hate speech directed at '
    'foreigners, immigrants, migrants, or people of a particular nationality or national origin.\n\n'
    'Label definitions:\n'
    '- XENOPHOBIC: the text expresses hostility, contempt, discrimination, dehumanization, '
    'exclusion, blame, or hatred against foreigners, immigrants, migrants, or people of a specific '
    'nationality or national origin.\n'
    '- NOT_XENOPHOBIC: the text does not contain xenophobic hate speech.\n\n'
    'Return only valid JSON in one line with this exact structure:\n'
    '{"label":"XENOPHOBIC"}\n'
    'or\n'
    '{"label":"NOT_XENOPHOBIC"}'
)

def make_request_line(row_id: int, text: str) -> dict:
    return {
        'custom_id': f'row-{row_id}',
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': MODEL_NAME,
            'temperature': TEMPERATURE,
            'max_tokens': MAX_TOKENS,
            'messages': [
                {'role': 'system', 'content': SYSTEM_MESSAGE},
                {'role': 'user',   'content': str(text)},
            ],
        },
    }

print('Helpers OK')

Helpers OK


---
## Chequeo previo — qué hay en OpenAI

Consulta el estado actual de cada batch conocido antes de decidir qué subir.

In [6]:
print(f'{"Parte":>6}  {"Batch ID":47}  {"Estado":<15}  Completados/Total')
print('-' * 92)

batch_status = {}
for part, batch_id in sorted(BATCH_REGISTRY.items()):
    if batch_id is None:
        batch_status[part] = 'not_submitted'
        print(f'{part:>6}  {"(nunca enviado)":47}  not_submitted')
    else:
        st  = client.batches.retrieve(batch_id)
        req = st.request_counts
        batch_status[part] = st.status
        print(f'{part:>6}  {batch_id:47}  {st.status:<15}  {req.completed}/{req.total}')

print()
needs = [p for p, s in batch_status.items() if s in ('not_submitted', 'failed', 'expired', 'cancelled')]
print(f'Partes que necesitan envío: {needs}')

 Parte  Batch ID                                         Estado           Completados/Total
--------------------------------------------------------------------------------------------
     1  batch_69d055d51dac8190b74033f1192fe9b9           completed        50000/50000
     2  batch_69d055ff99ac819092b4cdbc0316d286           completed        50000/50000
     3  batch_69d0562a9bd081908c953e14b591dcf5           completed        50000/50000
     4  batch_69d05655b2b0819089abbed186402091           completed        50000/50000
     5  batch_69d0567f64908190842c44876532c5bb           failed           0/0
     6  batch_69d05688d95481909361e6dfc7d7e688           failed           0/0

Partes que necesitan envío: [5, 6]


---
## Subir los batches faltantes

Solo procesa las partes detectadas arriba. Si no hay nada para subir, no hace nada.

In [7]:
if not needs:
    print('No hay partes para subir. Todo está enviado.')
else:
    JSONL_DIR.mkdir(parents=True, exist_ok=True)

    for part in needs:
        start   = (part - 1) * BATCH_SIZE
        end     = min(part * BATCH_SIZE, len(df))
        part_df = df.iloc[start:end]

        jsonl_path = JSONL_DIR / f'batch_requests_xenophobia_part_{part}.jsonl'
        with open(jsonl_path, 'w', encoding='utf-8') as fout:
            for _, row in part_df.iterrows():
                text = str(row[TEXT_COLUMN]) if pd.notna(row[TEXT_COLUMN]) else ''
                fout.write(json.dumps(make_request_line(int(row['row_id']), text), ensure_ascii=False) + '\n')

        size_mb = jsonl_path.stat().st_size / 1e6
        print(f'Subiendo parte {part:2d} ({len(part_df):,} filas, {size_mb:.1f} MB)...')
        input_file = client.files.create(file=open(jsonl_path, 'rb'), purpose='batch')
        batch = client.batches.create(
            input_file_id=input_file.id,
            endpoint='/v1/chat/completions',
            completion_window=COMPLETION_WINDOW,
            metadata={'job': f'xenophobia-part{part}'},
        )
        BATCH_REGISTRY[part] = batch.id
        batch_status[part]   = batch.status
        print(f'  -> {batch.id}  [{batch.status}]')

print()
print('=' * 60)
print('BATCH_REGISTRY FINAL — copiá esto al notebook de descarga')
print('=' * 60)
for p, bid in sorted(BATCH_REGISTRY.items()):
    print(f'    {p:2d}: {repr(bid)},')

Subiendo parte  5 (50,000 filas, 51.0 MB)...
  -> batch_69d166d415d88190b4b495b3c7f82008  [validating]
Subiendo parte  6 (9,752 filas, 9.9 MB)...
  -> batch_69d166de289c8190b8e294d1a2011c76  [validating]

BATCH_REGISTRY FINAL — copiá esto al notebook de descarga
     1: 'batch_69d055d51dac8190b74033f1192fe9b9',
     2: 'batch_69d055ff99ac819092b4cdbc0316d286',
     3: 'batch_69d0562a9bd081908c953e14b591dcf5',
     4: 'batch_69d05655b2b0819089abbed186402091',
     5: 'batch_69d166d415d88190b4b495b3c7f82008',
     6: 'batch_69d166de289c8190b8e294d1a2011c76',
